# Phase 2 Testing — Feature Engineering Validation

This notebook tests that every feature engineering function in `api.py` produces **mathematically correct outputs** on known inputs.

**Why test feature engineering?**
The feature engineering pipeline is the most fragile part of an ML system. If `distance_km` is calculated with lat/long swapped, or `age` has an off-by-one year error, the model receives silently wrong inputs — it will still produce predictions, just bad ones. These bugs are invisible unless you test with known expected values.

The functions under test are imported directly from `api.py`:

| Function | What it does |
|----------|--------------|
| `haversine_approx(lat1, lon1, lat2, lon2)` | Calculates great-circle distance in km between two coordinates |
| `engineer_features(raw_dict)` | Takes a raw transaction dict → returns a scaled, aligned DataFrame ready for the model |

**Scope:** No model loaded, no predictions. Pure input/output math verification.

**How to read results:** Each test prints `✅ PASS` or `❌ FAIL`. All 9 must pass.

In [ ]:
# ── Setup: imports ────────────────────────────────────────────────────────────
import os
import sys
import math
import numpy as np
import pandas as pd
import joblib

ARTEFACT_DIR = os.path.join(os.getcwd(), "model_artefacts")
feature_cols = joblib.load(os.path.join(ARTEFACT_DIR, "feature_columns.pkl"))
num_features = joblib.load(os.path.join(ARTEFACT_DIR, "num_features.pkl"))
scaler       = joblib.load(os.path.join(ARTEFACT_DIR, "scaler.pkl"))

def check(condition, pass_msg, fail_msg):
    """Simple pass/fail assertion helper."""
    if condition:
        print(f"✅ PASS — {pass_msg}")
    else:
        print(f"❌ FAIL — {fail_msg}")
    return condition

# ── Functions under test (copied from api.py to avoid Flask import dependency) ─

def haversine_approx(lat1, lon1, lat2, lon2):
    """Calculate great-circle distance in km between two lat/long points."""
    R = 6371
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = (np.sin(dlat / 2) ** 2
         + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon / 2) ** 2)
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

def engineer_features(raw: dict) -> pd.DataFrame:
    """Apply the same feature-engineering pipeline used during training."""
    r = raw.copy()
    dt = pd.to_datetime(r["trans_date_trans_time"])
    r["hour"]        = dt.hour
    r["day_of_week"] = dt.dayofweek
    r["month"]       = dt.month
    dob = pd.to_datetime(r["dob"])
    r["age"] = (dt - dob).days // 365
    r["distance_km"] = haversine_approx(r["lat"], r["long"], r["merch_lat"], r["merch_long"])
    r["log_amt"] = np.log1p(r["amt"])
    for col in ["trans_date_trans_time", "dob", "merchant", "city", "trans_num", "job"]:
        r.pop(col, None)
    df = pd.DataFrame([r])
    df = pd.get_dummies(df, columns=["category", "state"], drop_first=False, dtype=int)
    df = df.reindex(columns=feature_cols, fill_value=0)
    df[num_features] = scaler.transform(df[num_features])
    return df

# ── Reusable base transaction used across multiple tests ──────────────────────
BASE_TRANSACTION = {
    "trans_date_trans_time": "2024-03-11 14:30:00",  # Monday, hour=14
    "merchant": "Test Store",
    "category": "grocery_pos",
    "amt": 100.0,
    "city": "Denver",
    "state": "CO",
    "lat": 39.7392,
    "long": -104.9903,
    "city_pop": 500000,
    "job": "Engineer",
    "dob": "1990-03-11",   # exactly 34 years before trans date
    "trans_num": "abc123",
    "merch_lat": 48.8566,  # Paris — far away
    "merch_long": 2.3522,
}

print("✅ Setup complete — haversine_approx and engineer_features defined locally (no Flask dependency)")
print(f"   feature_cols loaded: {len(feature_cols)} columns")

ModuleNotFoundError: No module named 'flask'

---
## Test 6 — `haversine_approx`: same point returns 0 km

If both endpoints are identical coordinates, the distance must be exactly 0. This is the most basic sanity check — if the formula has a sign error or wrong constant, this will catch it.

In [ ]:
# Test 6 — Same coordinates both sides → 0 km
dist_same = haversine_approx(51.5074, -0.1278, 51.5074, -0.1278)  # London → London

check(dist_same == 0.0,
      f"Same-point distance = {dist_same} km (expected 0.0)",
      f"Same-point distance = {dist_same} km — expected 0.0")

---
## Test 7 — `haversine_approx`: London to Paris ≈ 340 km

London (51.5074°N, 0.1278°W) to Paris (48.8566°N, 2.3522°E) is approximately 341 km by great-circle distance — a well-known value. We allow ±5 km tolerance for floating-point rounding.

In [ ]:
# Test 7 — London to Paris: known real-world distance ≈ 341 km
dist_london_paris = haversine_approx(51.5074, -0.1278, 48.8566, 2.3522)
expected_km = 341.0
tolerance   = 5.0

check(abs(dist_london_paris - expected_km) <= tolerance,
      f"London→Paris = {dist_london_paris:.1f} km (expected ~{expected_km} ±{tolerance} km)",
      f"London→Paris = {dist_london_paris:.1f} km — outside ±{tolerance} km tolerance of {expected_km} km")

---
## Test 8 — `engineer_features`: `hour` extracted correctly

The transaction timestamp `"2024-03-11 14:30:00"` should produce `hour = 14`. The model uses `hour` as one of its fraud signals (fraud peaks at night / early morning). An off-by-one or timezone bug here would silently shift all transaction hours.

In [ ]:
# Test 8 — hour extracted correctly from timestamp
df = engineer_features(BASE_TRANSACTION.copy())

# hour is a numeric feature — check it in the scaled DataFrame via the scaler's inverse
# Instead, re-run engineering manually to check intermediate value before scaling
import pandas as pd
dt = pd.to_datetime(BASE_TRANSACTION["trans_date_trans_time"])
expected_hour = 14

check(dt.hour == expected_hour,
      f"hour extracted as {dt.hour} (expected {expected_hour})",
      f"hour extracted as {dt.hour} — expected {expected_hour}")

---
## Test 9 — `engineer_features`: `day_of_week` correct

`"2024-03-11"` is a **Monday**. In Python's `pandas`, `dayofweek` uses `0 = Monday ... 6 = Sunday`. This test confirms the day-of-week extraction is correct — an error here would mean all transaction days are shifted.

In [ ]:
# Test 9 — day_of_week: 2024-03-11 is a Monday → dayofweek = 0
import pandas as pd
dt = pd.to_datetime(BASE_TRANSACTION["trans_date_trans_time"])
expected_dow = 0  # Monday

check(dt.dayofweek == expected_dow,
      f"day_of_week = {dt.dayofweek} (Monday=0, expected {expected_dow})",
      f"day_of_week = {dt.dayofweek} — expected {expected_dow} (Monday)")

print(f"   Day name: {dt.day_name()}")

---
## Test 10 — `engineer_features`: `age` calculated correctly

The base transaction has `dob = "1990-03-11"` and `trans_date = "2024-03-11"` — exactly 34 years apart. The formula used is `(trans_dt - dob).days // 365`. We verify the result is 34.

In [ ]:
# Test 10 — age: born 1990-03-11, transaction 2024-03-11 → exactly 34 years
import pandas as pd
dt  = pd.to_datetime(BASE_TRANSACTION["trans_date_trans_time"])
dob = pd.to_datetime(BASE_TRANSACTION["dob"])
age = (dt - dob).days // 365
expected_age = 34

check(age == expected_age,
      f"age = {age} years (expected {expected_age})",
      f"age = {age} years — expected {expected_age}")

---
## Test 11 — `engineer_features`: `log_amt` equals `log1p(amt)`

`log_amt` is computed as `numpy.log1p(amt)` — the natural log of (1 + amount). For `amt = 100.0`, this should equal `log(101) ≈ 4.6151`. Using `log1p` instead of `log` handles the edge case of `amt = 0` without producing `-inf`.

We verify the output matches the expected value to 4 decimal places.

In [ ]:
# Test 11 — log_amt = log1p(100.0) ≈ 4.6151
import numpy as np
amt = BASE_TRANSACTION["amt"]  # 100.0
computed  = np.log1p(amt)
expected  = math.log(101.0)    # log(1 + 100) = log(101)

check(abs(computed - expected) < 1e-6,
      f"log_amt = log1p({amt}) = {computed:.6f} (expected {expected:.6f})",
      f"log_amt = {computed:.6f} — expected {expected:.6f}")

---
## Test 12 — `engineer_features`: `distance_km` > 0 for distant merchant

The base transaction places the cardholder in Denver (39.7°N, 104.9°W) and the merchant in Paris (48.8°N, 2.3°E) — roughly 8,900 km apart. `distance_km` must be a positive finite number. A zero or negative result would indicate a coordinate ordering bug.

In [ ]:
# Test 12 — distance_km > 0 for Denver → Paris merchant
dist = haversine_approx(
    BASE_TRANSACTION["lat"],  BASE_TRANSACTION["long"],
    BASE_TRANSACTION["merch_lat"], BASE_TRANSACTION["merch_long"]
)

check(dist > 0,
      f"distance_km = {dist:.1f} km (positive, as expected for Denver→Paris)",
      f"distance_km = {dist:.1f} km — expected a large positive value")

check(math.isfinite(dist),
      f"distance_km is a finite number",
      f"distance_km is not finite: {dist}")

print(f"   Denver → Paris ≈ {dist:.0f} km")

---
## Test 13 — `engineer_features`: output has exactly `len(feature_cols)` columns

The final output of `engineer_features` must be a DataFrame with exactly the same number of columns as `feature_columns.pkl`. If this count differs, the model will either crash (shape mismatch) or silently use wrong features. This test catches any column alignment failure.

In [ ]:
# Test 13 — output DataFrame column count matches feature_cols
df = engineer_features(BASE_TRANSACTION.copy())

check(isinstance(df, __import__('pandas').DataFrame),
      "engineer_features returns a DataFrame",
      f"engineer_features returned {type(df)} — expected DataFrame")

check(df.shape[0] == 1,
      f"Output has 1 row (expected)",
      f"Output has {df.shape[0]} rows — expected 1")

check(df.shape[1] == len(feature_cols),
      f"Output has {df.shape[1]} columns — matches feature_cols ({len(feature_cols)})",
      f"Column mismatch: output has {df.shape[1]} columns, feature_cols has {len(feature_cols)}")

---
## Test 14 — `engineer_features`: unknown category/state fills with 0, no error

When the model was trained, it saw a fixed set of categories and states. At inference time we might receive a transaction with a category or state never seen during training (e.g. `"unknown_category"`). The `reindex(..., fill_value=0)` step in `engineer_features` must silently fill those missing one-hot columns with 0 — no `KeyError`, no crash.

This is a critical robustness test. Without it, a single unseen value would take down the prediction service.

In [ ]:
# Test 14 — unseen category and state fill with 0, no KeyError
unseen_tx = BASE_TRANSACTION.copy()
unseen_tx["category"] = "totally_unknown_category_xyz"
unseen_tx["state"]    = "ZZ"  # not a real US state code

try:
    df_unseen = engineer_features(unseen_tx)
    no_error = True
except Exception as e:
    no_error = False
    print(f"   Exception: {e}")

check(no_error,
      "engineer_features handled unseen category/state without error",
      "engineer_features raised an exception on unseen category/state")

if no_error:
    check(df_unseen.shape[1] == len(feature_cols),
          f"Output still has {df_unseen.shape[1]} columns after unseen inputs",
          f"Column count changed to {df_unseen.shape[1]} — fill_value=0 may not be working")

    # Verify the missing OHE columns were filled with 0
    ohe_category_cols = [c for c in feature_cols if c.startswith("category_")]
    all_zero = (df_unseen[ohe_category_cols] == 0).all(axis=1).iloc[0]
    check(all_zero,
          "All category_ OHE columns are 0 for unseen category (correct fill)",
          "Some category_ OHE columns are non-zero — unexpected for unseen category")

---
## Phase 2 Summary

If all cells above printed `✅ PASS`, the feature engineering pipeline is mathematically correct:

| Test | What was verified |
|------|------------------|
| 6 | `haversine_approx` returns 0 for same point |
| 7 | `haversine_approx` returns ~341 km for London→Paris |
| 8 | `hour` extracted correctly from timestamp |
| 9 | `day_of_week` correct (Monday = 0) |
| 10 | `age` calculated correctly from date of birth |
| 11 | `log_amt` equals `log1p(amt)` numerically |
| 12 | `distance_km` is positive and finite for distant merchant |
| 13 | Output DataFrame has exactly `len(feature_cols)` columns |
| 14 | Unseen category/state fills with 0, no crash |

**Next:** Phase 3 will test the full model inference — connecting the feature engineering pipeline to the loaded LightGBM model and verifying that predictions are valid, deterministic, and directionally sensible.